# 24장 실험 기록과 재현성 — 숫자 하나를 주장으로 만들기

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `multiseed.py` 를 치고 실행. `multiseed.json` 을 열어 구조 확인 | [ch24_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `summarize` 의 각 줄에 통계 용어(평균·표준편차·표준오차·신뢰구간)를 주석으로 | [ch24_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_2_comment.ipynb) |
| 3. 다시 만들기 | 12장 `experiments.py` 를 씨앗 3개로 반복하도록 고쳐, 12.4의 표에 ± 를 붙이기 | [ch24_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "24"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/24")

준비 끝. 작업 폴더: code/24


## `multiseed.py` — 24장 같은 실험을 씨앗만 바꿔 여러 번: 점추정치에 불확실성을 붙인다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 24장 같은 실험을 씨앗만 바꿔 여러 번: 점추정치에 불확실성을 붙인다.
# 14장 패키지의 train.py 를 함수로 불러 쓰지 않고, 여기서는 12장 크기의 작은 실험을 자체 정의한다.
import json
import math
import os
import statistics
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "19"))
from statutes_dataset import StatuteDataset

LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
OUT = os.path.join(os.path.dirname(__file__), "results")
os.makedirs(OUT, exist_ok=True)


class BagOfWords(nn.Module):
    def __init__(self, vocab_size, dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, ids):
        vec = self.emb(ids)
        mask = (ids != 0).unsqueeze(-1).float()
        return self.fc((vec * mask).sum(1) / mask.sum(1).clamp(min=1))


def one_run(seed, lr, epochs=6):
    torch.manual_seed(seed)
    train_set = StatuteDataset(LAWS, "train", seed=seed)                  # 분할도 씨앗에 따라 달라진다
    test_set = StatuteDataset(LAWS, "test", vocab=train_set.vocab, seed=seed)
    loader = DataLoader(train_set, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=256)
    model = BagOfWords(len(train_set.vocab), 64, len(LAWS))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    model.eval()
    correct, per_class = 0, {c: [0, 0] for c in LAWS}
    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            for p, t in zip(pred.tolist(), y.tolist()):
                per_class[LAWS[t]][1] += 1
                per_class[LAWS[t]][0] += int(p == t)
    acc = correct / len(test_set)
    recall = {c: v[0] / v[1] for c, v in per_class.items()}                # 클래스별 재현율
    return {"seed": seed, "lr": lr, "acc": acc, "recall": recall, "n_test": len(test_set)}


def summarize(runs):
    accs = [r["acc"] for r in runs]
    m, s = statistics.mean(accs), statistics.stdev(accs) if len(accs) > 1 else 0.0
    ci = 1.96 * s / math.sqrt(len(accs))                                    # 정규 근사 95% 신뢰구간(씨앗 5개면 거칠다)
    return m, s, ci


if __name__ == "__main__":
    seeds = [0, 1, 2, 3, 4]
    all_runs = []
    for lr in (1e-3, 3e-3):
        runs = []
        t0 = time.time()
        for s in seeds:
            runs.append(one_run(s, lr))
        m, sd, ci = summarize(runs)
        print(f"lr={lr:<6} 정확도 {m:.3f} ± {ci:.3f} (95% CI, n={len(seeds)} 씨앗, SD {sd:.3f})  "
              f"개별 {[round(r['acc'], 3) for r in runs]}  ({time.time() - t0:.0f}초)")
        all_runs += runs
    # 클래스별 재현율(마지막 설정의 씨앗 평균): 정확도 하나로는 안 보이는 것
    last = [r for r in all_runs if r["lr"] == 3e-3]
    print("클래스별 재현율 (lr=3e-3, 씨앗 평균):",
          {c: round(statistics.mean(r["recall"][c] for r in last), 3) for c in LAWS})
    with open(os.path.join(OUT, "multiseed.json"), "w", encoding="utf-8") as f:
        json.dump(all_runs, f, ensure_ascii=False, indent=2)
    print("저장:", os.path.join(OUT, "multiseed.json"))
```

In [ ]:
!python multiseed.py

lr=0.001  정확도 0.700 ± 0.018 (95% CI, n=5 씨앗, SD 0.020)  개별 [0.675, 0.7, 0.721, 0.721, 0.686]  (23초)
lr=0.003  정확도 0.891 ± 0.012 (95% CI, n=5 씨앗, SD 0.014)  개별 [0.892, 0.878, 0.888, 0.914, 0.884]  (10초)
클래스별 재현율 (lr=3e-3, 씨앗 평균): {'민법': 0.976, '형법': 0.863, '민사소송법': 0.809, '형사소송법': 0.824}
저장: …\AI-Coding-for-Lawyers\code\24\results\multiseed.json


## 결과 파일

`multiseed.json` 의 구조를 확인합니다.

In [ ]:
print(open('results/multiseed.json', encoding='utf-8').read())

[
  {
    "seed": 0,
    "lr": 0.001,
    "acc": 0.6749049429657795,
    "recall": {
      "민법": 0.9818181818181818,
      "형법": 0.7375,
      "민사소송법": 0.22727272727272727,
      "형사소송법": 0.47413793103448276
    },
    "n_test": 526
  },
  {
    "seed": 1,
    "lr": 0.001,
    "acc": 0.6996197718631179,
    "recall": {
      "민법": 0.9913793103448276,
      "형법": 0.676056338028169,
      "민사소송법": 0.38392857142857145,
      "형사소송법": 0.42342342342342343
    },
    "n_test": 526
  },
  {
    "seed": 2,
    "lr": 0.001,
    "acc": 0.720532319391635,
    "recall": {
      "민법": 0.9909502262443439,
      "형법": 0.7407407407407407,
      "민사소송법": 0.3786407766990291,
      "형사소송법": 0.5041322314049587
    },
    "n_test": 526
  },
  {
    "seed": 3,
    "lr": 0.001,
    "acc": 0.720532319391635,
    "recall": {
      "민법": 0.9857142857142858,
      "형법": 0.7395833333333334,
      "민사소송법": 0.3564356435643564,
      "형사소송법": 0.5462184873949579
    },
    "n_test": 526
  },
  {
    "seed": 4,
    "l

## 다음 단계

- 2단계 「주석 달기」: [ch24_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch24_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch24_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.